## Goldwork_Incremental
incremental Gold processing plus latest and timestamped Volume sanpshots

## Imports and setup

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from datetime import datetime, UTC

In [0]:
spark.sql("use catalog novacart_casestudy")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().strftime("%Y-%m-%d")


print("Cuurent Gold Run ID", gold_run_id)
print("Run Timestamp Folder", run_ts_str)



### Gold control Table

In [0]:
spark.sql("DROP TABLE IF EXISTS novacart_casestudy.gold_schema.processing_control")

In [0]:
spark.sql("""
CREATE TABLE novacart_casestudy.gold_schema.processing_control (
    layer STRING,
    entity_name STRING,
    last_processed_silver_run_id STRING,
    last_processed_silver_run_ts TIMESTAMP,
    rows_merged BIGINT,
    run_status STRING,
    gold_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

### Helper functions
- **upsert_to_gold()** merges data into Gold current-state tables
- **get_last_processed_silver_ts()****reads the gold watermark from the control table
- **upsert_gold_control()** updates gold control after a successful run

In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl = (
        spark.table("novacart_casestudy.gold_schema.processing_control")
                .filter(
                    (F.col("layer") == "gold") &
                    (F.col("entity_name") == entity_name) &
                    (F.col("run_status") == 'SUCCESS')
                )
                .orderBy (F.col("updated_at").desc())
                .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None

    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(
    entity_name,
    last_processed_silver_run_id,
    last_processed_silver_run_ts,
    rows_merged
    
):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_run_ts,
            int(rows_merged),
            "SUCCESS",
            gold_run_id,
            datetime.now(UTC).replace(tzinfo=None)
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "novacart_casestudy.gold_schema.processing_control"
    )

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "s.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("last Processed silver Timestamp for gold=", last_gold_ts)

silver_orders_current = spark.read.table("novacart_casestudy.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_casestudy.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_casestudy.silver_schema.payments_transformed")

if last_gold_ts is None:

    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:

    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("updated_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed_orders= {changed_orders_count}")
print(f"Number of changed_products= {changed_products_count}")
print(f"Number of changed_payments= {changed_payments_count}")

    




### Find impacted order IDs
Gold is bulit at order grain,so if anything changes in orders,products or payments, we identify which order_id values are impacted

Only those order IDs are rebuilt in Gold

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"), 'inner').select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print("impacted_orders_ids =",impacted_order_ids.count() )
display(impacted_order_ids.orderBy("order_id"))


### Build Gold DELTA FOR IMPACTED ORDERS
this cell joins the impacted orders with the current silver products and payments tables,derives bussinescolumns and builds the 
gold delta thatwill be merged into Gold current-state table

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("pay"),
        F.col("o.order_id") == F.col("pay.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("pay.payment_id"),
        F.col("pay.payment_status"),
        F.col("pay.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("pay.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
         .when(F.col("payment_completion_ratio") == 0, "Unpaid")
         .when(F.col("payment_completion_ratio") == 1, "Paid")
         .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
         .when(F.col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)

print("gold_delta_rows =", gold_delta.count())
display(gold_delta)

### Merge Gold current-state table

In [0]:
if gold_delta.count() > 0:
  upsert_to_gold(gold_delta,"novacart_casestudy.gold_schema.orders_information","order_id")
else:
  print("No new rows to insert in gold table")

In [0]:
%sql
select * from novacart_casestudy.gold_schema.orders_information;

### Maintain Gold SCD Type 2 history

In [0]:
gold_delta_count = gold_delta.count()

spark.sql("DROP TABLE IF EXISTS novacart_casestudy.gold_schema.orders_information_scd2")

spark.sql("""
    CREATE TABLE novacart_casestudy.gold_schema.orders_information_scd2
    USING DELTA
    AS
    SELECT *,
           CAST(NULL AS TIMESTAMP) AS valid_from_ts,
           CAST(NULL AS TIMESTAMP) AS valid_to_ts,
           TRUE AS is_current
    FROM novacart_casestudy.gold_schema.orders_information
    WHERE 1 = 0
""")

if gold_delta_count > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")

    spark.sql("""
        MERGE INTO novacart_casestudy.gold_schema.orders_information_scd2 t
        USING gold_delta_view s
        ON t.order_id = s.order_id AND t.is_current = true
        WHEN MATCHED AND (
            NOT (t.order_status <=> s.order_status) OR
            NOT (t.order_amount <=> s.order_amount) OR
            NOT (t.paid_amount <=> s.paid_amount) OR
            NOT (t.payment_id <=> s.payment_id) OR
            NOT (t.category <=> s.category) OR
            NOT (t.product_name <=> s.product_name) OR
            NOT (t.product_price <=> s.product_price)
        )
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to_ts = s.gold_update_ts
    """)

    spark.sql("""
        INSERT INTO novacart_casestudy.gold_schema.orders_information_scd2
        SELECT
            s.*,
            s.gold_update_ts AS valid_from_ts,
            CAST(NULL AS TIMESTAMP) AS valid_to_ts,
            TRUE AS is_current
        FROM gold_delta_view s
        LEFT JOIN novacart_casestudy.gold_schema.orders_information_scd2 t
            ON s.order_id = t.order_id
           AND t.is_current = true
        WHERE t.order_id IS NULL
           OR (
                NOT (t.order_status <=> s.order_status) OR
                NOT (t.order_amount <=> s.order_amount) OR
                NOT (t.paid_amount <=> s.paid_amount) OR
                NOT (t.payment_id <=> s.payment_id) OR
                NOT (t.category <=> s.category) OR
                NOT (t.product_name <=> s.product_name) OR
                NOT (t.product_price <=> s.product_price)
           )
    """)

### Update category-level Gold aggregation
This cell recalculates category-level bussiness metrics only for categories impacted in the current run,then merges them into the category performance Gold table

In [0]:
if gold_delta.count() > 0:
    impacted_categories =(
        gold_delta.select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )

    category_pref_delta = (
        spark.read.table("novacart_casestudy.gold_schema.orders_information")
        .join(impacted_categories,"category","inner")
        .groupBy("category")
        .agg(F.countDistinct("order_id").alias("total_orders"),
             F.sum(
                 F.when(F.col("order_amount") > 0, F.col("order_amount"))
                 .otherwise(F.lit(0.0))
             ).alias("Gross_Merchandise_Value"),
             F.sum(
                 F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                 .otherwise(F.lit(0.0))
             ).alias("Total_amount_paid"),
             F.avg(F.col("payment_completion_ratio")).alias("Avg_payment_completion_ratio"),
             (
             F.sum(F.when(F.col("payment_status") == 'FAILED',1).otherwise(0))/F.count("*")
             ).alias("payment_Failure_Rate")
             
         )
    )
    upsert_to_gold(category_pref_delta,"novacart_casestudy.gold_schema.category_performance","category")

In [0]:
%sql
select * from novacart_casestudy.gold_schema.category_performance;

In [0]:
%sql
select * from novacart_casestudy.gold_schema.category_performance;

### Publish Gold sanpshots to Volume
This cell writes two kinds of gold outputs to a databricks volume:
lastest snapshots - overwritten every successful run
timestamped historical snapshot- a new folder for each successful run

This is useful for aduit,roolback,teaching demos


In [0]:
spark.sql("create volume if not exists novacart_casestudy.gold_schema.gold_snapshots_vol")

In [0]:
run_ts_safe = run_ts_str.replace(" ", "_").replace(":", "-")

latest_orders_path = (
    "/Volumes/novacart_casestudy/gold_schema/gold_snapshots_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/novacart_casestudy/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)

historical_orders_path = (
    f"/Volumes/novacart_casestudy/gold_schema/gold_snapshots_vol/"
    f"gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_safe}"
)

historical_category_path = (
    f"/Volumes/novacart_casestudy/gold_schema/gold_snapshots_vol/"
    f"gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_safe}"
)

spark.read.table("novacart_casestudy.gold_schema.orders_information") \
    .write.mode("overwrite").format("parquet").save(latest_orders_path)

spark.read.table("novacart_casestudy.gold_schema.category_performance") \
    .write.mode("overwrite").format("parquet").save(latest_category_path)
spark.read.table("novacart_casestudy.gold_schema.orders_information") \
    .write.mode("overwrite").format("parquet").save(historical_orders_path)

spark.read.table("novacart_casestudy.gold_schema.category_performance") \
    .write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest Orders path:", latest_orders_path)
print("latest Category path:", latest_category_path)
print("historical Orders path:", historical_orders_path)
print("historical Category path:", historical_category_path)

### update gold control table
This final cell updates the gold control table using latest silver processing metadata and display the control table for validation

In [0]:
from datetime import datetime, timezone
datetime.now(timezone.utc)
last_processed_silver_run_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]


latest_silver_run_id =(
  silver_orders_current
  .filter(F.col("bronze_ingested_at") == last_processed_silver_run_ts)
  .agg(F.max("silver_run_id").alias("mx"))
  .collect()[0]["mx"]

) if last_processed_silver_run_ts is not None else None

upsert_gold_control("orders_information",last_processed_silver_run_id,last_processed_silver_run_ts,gold_delta.count())
display(spark.table("novacart_casestudy.gold_schema.processing_control"))

In [0]:


latest_silver_ts = (
    silver_orders_current
    .agg(F.max("bronze_ingested_at").alias("mx"))
    .collect()[0]["mx"]
)

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == last_processed_silver_run_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_run_ts is not None else None

upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_run_ts,
    gold_delta.count()
    
)

display(spark.table("novacart_casestudy.gold_schema.processing_control"))


In [0]:
latest_silver_ts = (
    silver_orders_current
    .agg(F.max("bronze_ingested_at").alias("mx"))
    .collect()[0]["mx"]
)

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta.count()
)

display(spark.table("novacart_casestudy.gold_schema.processing_control"))